# Notebook 2: Beta Prediction Models
**AdaptiveBeta — AI-Powered Portfolio Optimisation**  
**Author:** Kunal | M.Tech AI & ML, Symbiosis Institute of Technology, Pune

This notebook trains and compares 4 models for predicting 20-day forward beta volatility:

| Model | Type | Notes |
|-------|------|-------|
| Rolling OLS | Baseline | Fixed 60d window |
| Kalman Filter | State-space | Time-varying beta |
| XGBoost | Gradient boosting | + SHAP explainability |
| **LSTM** | **Deep learning** | **Proposed model** |

**Key rule:** NEVER shuffle time series — always chronological train/test split.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/AI_Finance_Project'

REPO = '/content/drive/MyDrive/AI_Finance_Project/repo'
if os.path.exists(REPO):
    sys.path.insert(0, REPO)

print('Drive mounted.')

## 2.1 — Load Features & Train/Test Split

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

TRAIN_END  = '2021-12-31'
TEST_START = '2022-01-01'

stacked = pd.read_csv(f'{ROOT}/features/stacked_features.csv', parse_dates=['date']).set_index('date')
feature_cols = [c for c in stacked.columns if c not in ['target', 'ticker']]

print(f'Stacked matrix: {stacked.shape}')
print(f'Feature columns ({len(feature_cols)}): {feature_cols[:5]} ...')

train_df = stacked[stacked.index <= TRAIN_END]
test_df  = stacked[stacked.index >= TEST_START]

X_train = train_df[feature_cols].values
y_train = train_df['target'].values
X_test  = test_df[feature_cols].values
y_test  = test_df['target'].values

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train date range: {train_df.index[0].date()} → {train_df.index[-1].date()}')
print(f'Test  date range: {test_df.index[0].date()}  → {test_df.index[-1].date()}')

## 2.2 — Model A: XGBoost Baseline

In [ ]:
import xgboost as xgb
import shap

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
)

xgb_model.fit(
    X_train_sc, y_train,
    eval_set=[(X_test_sc, y_test)],
    verbose=50,
)

xgb_pred = xgb_model.predict(X_test_sc)
xgb_mae  = mean_absolute_error(y_test, xgb_pred)

# Directional accuracy
y_diff = np.diff(y_test)
p_diff = np.diff(xgb_pred)
xgb_dir = np.mean(np.sign(p_diff) == np.sign(y_diff))

print(f'\nXGBoost — MAE: {xgb_mae:.4f} | Direction Accuracy: {xgb_dir:.3f}')

In [ ]:
# SHAP feature importance
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_sc)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)

print('Top 10 features by SHAP importance:')
print(importance_df.head(10).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
top15 = importance_df.head(15)
ax.barh(top15['feature'][::-1], top15['mean_abs_shap'][::-1], color='#1D9E75')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('XGBoost — Top 15 Feature Importances (SHAP)')
plt.tight_layout()
plt.savefig(f'{ROOT}/results/shap_importance.png', dpi=120, bbox_inches='tight')
plt.show()

importance_df.to_csv(f'{ROOT}/results/shap_importance.csv', index=False)

## 2.3 — Model B: Kalman Filter (Classical Comparison)

In [ ]:
from pykalman import KalmanFilter

# Load aligned returns
betavol_60 = pd.read_csv(f'{ROOT}/features/betavol_60d.csv', parse_dates=['date']).set_index('date')

# We'll compute Kalman betas for all stocks using the src module
# (This may take ~5 minutes for 49 stocks)
try:
    sys.path.insert(0, REPO)
    from src.models.kalman import kalman_beta_panel

    # Load returns
    prices_df = pd.read_csv(f'{ROOT}/raw_data/stocks/all_stocks_prices.csv',
                             parse_dates=['date']).set_index('date').sort_index()
    nifty_df  = pd.read_csv(f'{ROOT}/raw_data/market/nifty50.csv',
                             parse_dates=['date']).set_index('date').sort_index()
    TICKERS = [c for c in prices_df.columns if c != 'TMPV.NS']
    sr = np.log(prices_df[TICKERS] / prices_df[TICKERS].shift(1))
    mr = np.log(nifty_df['Close'] / nifty_df['Close'].shift(1))
    common_idx = sr.dropna(how='all').index.intersection(mr.dropna().index)
    sr = sr.loc[common_idx]; mr = mr.loc[common_idx]

    print('Computing Kalman betas (49 stocks × Kalman EM)...')
    kalman_betas_panel = kalman_beta_panel(sr, mr)
    kalman_betas_panel.to_csv(f'{ROOT}/features/kalman_betas.csv')
    print(f'Kalman betas saved — shape: {kalman_betas_panel.shape}')

    # Kalman betavol
    kalman_betavol = kalman_betas_panel.rolling(60).std()
    kalman_target  = kalman_betavol.shift(-20)

    # MAE on test period
    test_mask  = kalman_target.index >= TEST_START
    kalman_mae = kalman_target[test_mask].sub(
        betavol_60[test_mask].reindex(columns=kalman_target.columns)
    ).abs().mean().mean()
    print(f'Kalman — MAE: {kalman_mae:.4f}')

except Exception as e:
    print(f'Kalman computation: {e}')
    kalman_mae = np.nan
    print('Using NaN for Kalman MAE in comparison table')

## 2.4 — Model C: LSTM (Primary Deep Model)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Hyperparameters
SEQ_LEN     = 30
HIDDEN_SIZE = 64
N_LAYERS    = 2
DROPOUT     = 0.2
BATCH_SIZE  = 256
EPOCHS      = 50
LR          = 1e-3
PATIENCE    = 10

torch.manual_seed(42)
np.random.seed(42)

class BetaDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.seq_len = seq_len
    def __len__(self):
        return len(self.y) - self.seq_len
    def __getitem__(self, idx):
        return self.X[idx:idx+self.seq_len], self.y[idx+self.seq_len]

class BetaLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, n_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers,
                             batch_first=True, dropout=dropout if n_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.norm(out[:, -1, :])
        return self.head(out).squeeze(-1)

# Chronological val split (last 20% of train)
val_split   = int(len(X_train_sc) * 0.8)
X_tr, X_vl  = X_train_sc[:val_split], X_train_sc[val_split:]
y_tr, y_vl  = y_train[:val_split],    y_train[val_split:]

train_ds = BetaDataset(X_tr, y_tr, SEQ_LEN)
val_ds   = BetaDataset(X_vl, y_vl, SEQ_LEN)
test_ds  = BetaDataset(X_test_sc, y_test, SEQ_LEN)

# CRITICAL: shuffle=False — never shuffle time series
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

n_features = len(feature_cols)
model = BetaLSTM(n_features, HIDDEN_SIZE, N_LAYERS, DROPOUT).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, patience=5, factor=0.5)
loss_fn = nn.HuberLoss()

best_val_loss = float('inf')
patience_ctr  = 0
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0.0
    for X_b, y_b in train_dl:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optim.zero_grad()
        pred = model(X_b)
        loss = loss_fn(pred, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_b, y_b in val_dl:
            X_b, y_b = X_b.to(device), y_b.to(device)
            val_loss += loss_fn(model(X_b), y_b).item()

    tr_avg  = train_loss / max(len(train_dl), 1)
    vl_avg  = val_loss   / max(len(val_dl), 1)
    train_losses.append(tr_avg)
    val_losses.append(vl_avg)
    sched.step(vl_avg)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d} | Train: {tr_avg:.4f} | Val: {vl_avg:.4f} | '
              f'LR: {optim.param_groups[0]["lr"]:.1e}')

    if vl_avg < best_val_loss:
        best_val_loss = vl_avg
        torch.save(model.state_dict(), f'{ROOT}/models/lstm_best.pt')
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})')
            break

print('Training complete.')

In [ ]:
# Load best model and evaluate on test set
model.load_state_dict(torch.load(f'{ROOT}/models/lstm_best.pt', map_location=device))
model.eval()

lstm_preds = []
with torch.no_grad():
    for X_b, _ in test_dl:
        out = model(X_b.to(device))
        lstm_preds.extend(out.cpu().numpy().tolist())

lstm_pred = np.array(lstm_preds)
y_test_aligned = y_test[SEQ_LEN:SEQ_LEN + len(lstm_pred)]

lstm_mae = mean_absolute_error(y_test_aligned, lstm_pred)
y_diff = np.diff(y_test_aligned)
p_diff = np.diff(lstm_pred)
valid  = y_diff != 0
lstm_dir = np.mean(np.sign(p_diff[valid]) == np.sign(y_diff[valid]))

print(f'LSTM — MAE: {lstm_mae:.4f} | Direction Accuracy: {lstm_dir:.3f}')

# Plot training curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label='Train Loss', color='#1D9E75')
ax.plot(val_losses,   label='Val Loss',   color='#7F77DD')
ax.set_xlabel('Epoch'); ax.set_ylabel('Huber Loss')
ax.set_title('LSTM Training Curves')
ax.legend()
plt.tight_layout()
plt.savefig(f'{ROOT}/results/lstm_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.5 — Model D: HMM Regime Classifier

In [ ]:
from hmmlearn import hmm

# Load market returns and VIX for regime detection
nifty_raw = pd.read_csv(f'{ROOT}/raw_data/market/nifty50.csv', parse_dates=['date']).set_index('date').sort_index()
vix_raw   = pd.read_csv(f'{ROOT}/raw_data/market/india_vix.csv', parse_dates=['date']).set_index('date').sort_index()

mr = np.log(nifty_raw['Close'] / nifty_raw['Close'].shift(1)).dropna()
vx = vix_raw['Close'].reindex(mr.index).ffill().fillna(method='bfill')
rv = mr.rolling(5).std()

hmm_df = pd.DataFrame({'mr': mr, 'vix': vx, 'vol5': rv}).dropna()
X_hmm = hmm_df.values
X_hmm_norm = (X_hmm - X_hmm.mean(0)) / (X_hmm.std(0) + 1e-8)

hmm_model = hmm.GaussianHMM(n_components=3, covariance_type='full', n_iter=100, random_state=42)
hmm_model.fit(X_hmm_norm)
states = hmm_model.predict(X_hmm_norm)

# Map states to labels by mean return
state_means = {s: hmm_df['mr'][states == s].mean() for s in range(3)}
sorted_states = sorted(state_means, key=state_means.get)
label_map = {sorted_states[0]: 'bear', sorted_states[1]: 'transition', sorted_states[2]: 'bull'}

regime_labels = pd.Series([label_map[s] for s in states], index=hmm_df.index, name='regime')
regime_labels.to_csv(f'{ROOT}/features/hmm_regimes.csv')

print('Regime distribution:')
print(regime_labels.value_counts())
print('Saved: hmm_regimes.csv')

In [ ]:
# Visualise regime classification
fig, ax = plt.subplots(figsize=(14, 4))
colors = {'bull': '#1D9E75', 'bear': '#D85A30', 'transition': '#888780'}
cum_nifty = np.exp(mr.cumsum()) * 100

prev_date, prev_regime = regime_labels.index[0], regime_labels.iloc[0]
for date, regime in regime_labels.items():
    if regime != prev_regime:
        ax.axvspan(prev_date, date, alpha=0.2, color=colors.get(prev_regime, 'gray'))
        prev_date, prev_regime = date, regime
ax.axvspan(prev_date, regime_labels.index[-1], alpha=0.2, color=colors.get(prev_regime, 'gray'))

cum_nifty.reindex(regime_labels.index).plot(ax=ax, color='white', lw=1.5, label='NIFTY50')
ax.set_title('HMM Market Regime Classification')
ax.legend()
plt.tight_layout()
plt.savefig(f'{ROOT}/results/hmm_regimes.png', dpi=120, bbox_inches='tight')
plt.show()

## 2.6 — Model Comparison Table

In [ ]:
# Rolling OLS baseline: predict 60d betavol using last observed value
betavol_60 = pd.read_csv(f'{ROOT}/features/betavol_60d.csv', parse_dates=['date']).set_index('date')
ols_baseline = betavol_60[betavol_60.index >= TEST_START]
ols_mae = ols_baseline.sub(
    pd.read_csv(f'{ROOT}/features/target_betavol_20d_ahead.csv',
                 parse_dates=['date']).set_index('date')[ols_baseline.index[0]:][ols_baseline.columns]
).abs().mean().mean()

results_table = pd.DataFrame({
    'Model':         ['Static OLS Beta (baseline)', 'Kalman Filter', 'XGBoost', 'LSTM (proposed)'],
    'MAE':           [round(ols_mae, 4), round(float(kalman_mae) if not np.isnan(kalman_mae) else np.nan, 4),
                      round(xgb_mae, 4), round(lstm_mae, 4)],
    'Direction_Acc': [np.nan, np.nan, round(xgb_dir, 3), round(lstm_dir, 3)],
    'Notes':         ['Fixed 60d rolling OLS', 'State-space EM', 'Gradient boosting + SHAP', 'Seq2One LSTM + LayerNorm'],
})

results_table.to_csv(f'{ROOT}/results/model_comparison.csv', index=False)
print('Model Comparison:')
print(results_table.to_string(index=False))

In [ ]:
# Save scaler for use in Notebook 3 & 4
import pickle
with open(f'{ROOT}/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature column list
with open(f'{ROOT}/models/feature_cols.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print('Saved: scaler.pkl, feature_cols.txt, lstm_best.pt')
print('\n✅ Notebook 2 complete — proceed to Notebook 3: Portfolio Optimisation')